<a href="https://colab.research.google.com/github/lohithrchandu/skill-lab-a2/blob/main/Copy_of_agent_loop_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [ ]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

In [ ]:
!pip install -U google-genai --quiet
print("Done.")

In [ ]:
# @title User input needed.

# TODO: Fill in documents in the format mentioned in the first example.
DOCUMENTS = {
    "sample_protocol": (
        "Students must fill in documents in this section. "
        "Documents can be related to any topic - sports, manga, food, music etc. "
        "Have a minimum of 4 sentences with each document. "
        "Duplicate this for a minimum of 4 documents to test your agent."
    ),
    "valorant": (
        "Valorant is a free-to-play tactical shooter developed by Riot Games. "
        "Two teams of five players face off, with one side attacking and the other defending. "
        "Attackers try to plant a bomb called the spike, while defenders try to stop them or defuse it. "
        "Each player picks an agent with unique abilities, which adds strategy on top of the shooting. "
        "A match is won by the first team to win 13 rounds."
    ),
    "masala_dosa": (
        "Masala dosa is a popular South Indian dish made from a fermented batter of rice and urad dal. "
        "The batter is spread thin on a hot griddle until it turns golden and crisp. "
        "It is filled with a spiced potato mixture made with onions, mustard seeds and curry leaves. "
        "Masala dosa is usually served with sambar and coconut chutney. "
        "It is eaten for breakfast across India and is also a common restaurant meal."
    ),
    "one_piece": (
        "One Piece is a manga series written and illustrated by Eiichiro Oda. "
        "It began serialization in Weekly Shonen Jump in 1997 and is one of the best-selling manga series ever. "
        "The story follows Monkey D. Luffy, a young pirate who gained rubber-like powers after eating a Devil Fruit. "
        "Luffy leads the Straw Hat Pirates across the Grand Line in search of the treasure called One Piece. "
        "His goal is to become the Pirate King."
    ),
    "ar_rahman": (
        "A. R. Rahman is an Indian composer, singer and music producer from Chennai. "
        "He first became famous for his film scores in Tamil cinema in the early 1990s. "
        "He won two Academy Awards for the film Slumdog Millionaire, including Best Original Song for Jai Ho. "
        "His music blends Indian classical, electronic and world music styles. "
        "He is known for using technology and layered sound design in his compositions."
    ),
}

In [ ]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_documents("minimum of 4"))

In [ ]:
# @title Model settings
MODEL_NAME = "gemini-3-flash-preview"

SYSTEM_INSTRUCTION = """
You are a helpful assistant that answers questions using a small
collection of documents. When you need facts, use the search_documents
tool. Only answer based on what you find in the documents.
If the documents do not contain the answer, say so clearly.
"""

print("Using model:", MODEL_NAME)

In [ ]:
import os
import time
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_documents",
        description="Search the documents to find the most relevant paragraph.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    system_instruction=SYSTEM_INSTRUCTION,
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=MODEL_NAME,
                contents=contents, config=config
            )
        except Exception as e:
            # Retry on rate limits (429) and temporary overloads (503)
            is_transient = "429" in str(e) or "503" in str(e) or "UNAVAILABLE" in str(e).upper()
            if is_transient and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Temporary server error or rate limit hit, waiting {wait}s before retrying...")
                time.sleep(wait)
            else:
                raise


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content
        parts = model_content.parts or []

        # Append the model's turn UNMODIFIED, so thought signatures are preserved
        contents.append(model_content)

        calls = [p.function_call for p in parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Run every requested tool call, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_documents(**args)
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"


_ = run_agent("What topics are students supposed to share documents on?")

In [ ]:
# @title Ask follow up questions

# TODO: Enter questions based on documents for the agent to answer.
questions = [
    "How many rounds does a team need to win a Valorant match?",
    "What is masala dosa usually served with?",
    "Who wrote One Piece and what is Luffy's goal?",
    # This question is not answered by the database.
    "How many ghosts eat carrots everyday?",
    # This question is part of the general knowledge of the model.
    "What is the most popular food item in Canada?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)

In [ ]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)